In [2]:
import pandas as pd
import numpy as np


In [47]:
data = pd.read_csv('../data/clean_data.csv')
print(f'''Список отслеживаемых событий: 
{data['EventName'].unique()}''')

Список отслеживаемых событий: 
['Tutorial' 'MainScreenAppear' 'OffersScreenAppear' 'CartScreenAppear'
 'PaymentScreenSuccessful']


In [40]:
data = data.sort_values(by =['DeviceIDHash','datetime'])
data.head(1)

,EventName,DeviceIDHash,ExpId,datetime,date
194435,MainScreenAppear,6888746892508752,246,2019-08-06 14:06:34,2019-08-06


In [60]:
# Количество событий без учета групп и количество уникальных пользовтелей по каждому событию
print(data.groupby('EventName').agg(cnt_event = ('EventName', 'count'), cnt_uniq_users = ('DeviceIDHash', 'nunique')).sort_values('cnt_event',ascending=False).reset_index())

# А сколько всего уникальных пользователей
print(f''' 
Уникальных пользователей: {data['DeviceIDHash'].nunique()}''')

# Уникальных > Зашедших на стартовую страницу!



                 EventName  cnt_event  cnt_uniq_users
0         MainScreenAppear     117431            7419
1       OffersScreenAppear      46350            4593
2         CartScreenAppear      42365            3734
3  PaymentScreenSuccessful      34113            3539
4                 Tutorial       1039             840
 
Уникальных пользователей: 7534


In [71]:
# Почему количество уникальных не совпадает с количество уникальных, видивших стартовый экран приложения

# перепроверим 
main_users = set(data[data['EventName'] == 'MainScreenAppear']['DeviceIDHash']) # уникальные пользователи, имеющие события MainScreenAppear
all_users = set(data['DeviceIDHash']) # все уникальные пользователи
not_main_users = all_users - main_users
print(f'Пользователи, которые не видели MainScreenAppear: {len(not_main_users)}')

# фильтруем пользователей "без основного экрана"
not_main_events = data[data['DeviceIDHash'].isin(not_main_users)]

# какие события встречаются у данных пользователей
not_main_users_events = not_main_events.groupby('DeviceIDHash').agg(
    event_list=('EventName', list),
    expid=('ExpId', 'first'),
    events_count=('EventName', 'count')  
).reset_index()

print("Пользователи, не дошедшие до MainScreenAppear:")
print(not_main_users_events)


# пользователи без покупок
not_payment = not_main_users_events[~not_main_users_events['event_list'].apply(lambda x: 'PaymentScreenSuccessful' in x)]
print(f'Пришедших извне и не сделавших покупок {len(not_payment)}')
not_payment


Пользователи, которые не видели MainScreenAppear: 115
Пользователи, не дошедшие до MainScreenAppear:
            DeviceIDHash                                         event_list  \
0      74158328448226259  [PaymentScreenSuccessful, CartScreenAppear, Of...   
1     111394506613435756  [PaymentScreenSuccessful, CartScreenAppear, Of...   
2     214966247576341063  [OffersScreenAppear, PaymentScreenSuccessful, ...   
3     261817378841141406  [PaymentScreenSuccessful, CartScreenAppear, Of...   
4     332529825412858125  [OffersScreenAppear, CartScreenAppear, OffersS...   
..                   ...                                                ...   
110  8586953157808767383  [OffersScreenAppear, CartScreenAppear, Payment...   
111  8804319115517716344  [PaymentScreenSuccessful, CartScreenAppear, Of...   
112  8821171531680573201  [OffersScreenAppear, PaymentScreenSuccessful, ...   
113  9124766629178994679  [OffersScreenAppear, OffersScreenAppear, Offer...   
114  9217594193087726423  [Pay

,DeviceIDHash,event_list,expid,events_count
12,1223708690315846789,[OffersScreenAppear],246,1
15,1478347681767261393,"[OffersScreenAppear, OffersScreenAppear, Offer...",247,4
19,1900791869709139147,[OffersScreenAppear],248,1
20,1958496982439584534,[OffersScreenAppear],248,1
25,2472435690120708424,[OffersScreenAppear],246,1
27,2525867977967066505,"[Tutorial, Tutorial, Tutorial]",247,3
34,2974131200515436842,"[OffersScreenAppear, OffersScreenAppear]",248,2
43,3800345857856674685,"[Tutorial, Tutorial]",248,2
48,4111480625662873403,"[OffersScreenAppear, CartScreenAppear, CartScr...",247,3
57,4530866328707864508,[Tutorial],248,1


In [85]:
# пришедших извне в каждой группе А/А/В
print(not_main_users_events.groupby('expid')['DeviceIDHash'].nunique())

expid
246    34
247    37
248    44
Name: DeviceIDHash, dtype: int64


In [86]:
# малоактивные пользователи
counts = data.groupby('DeviceIDHash')['EventName'].transform('count')
one_events = data[counts <= 2]
#print(one_events)
print(' Уникальных неактивных пользователей')
print(one_events['DeviceIDHash'].nunique())
print(f' С разбивкой по группам {one_events.groupby('ExpId')['DeviceIDHash'].nunique()}')
#one_events.groupby('EventName').agg(users = ('DeviceIDHash', 'nunique'), cnt_event = ('EventName', 'count'))


 Уникальных неактивных пользователей
282
 С разбивкой по группам ExpId
246    92
247    94
248    96
Name: DeviceIDHash, dtype: int64


In [5]:
data.groupby(['ExpId', 'EventName']).agg(cnt_event = ('EventName', 'count'), )

cnt_event
ExpId EventName                         
246   CartScreenAppear             14711
      MainScreenAppear             37708
      OffersScreenAppear           14773
      PaymentScreenSuccessful      11910
      Tutorial                       323
247   CartScreenAppear             12456
      MainScreenAppear             39123
      OffersScreenAppear           15182
      PaymentScreenSuccessful      10043
      Tutorial                       343
248   CartScreenAppear             15198
      MainScreenAppear             40600
      OffersScreenAppear           16395
      PaymentScreenSuccessful      12160
      Tutorial                       373

In [33]:
# Анализ последовательности событий на примере пользователя 6922444491712477
data.query('DeviceIDHash == 214966247576341063')

,EventName,DeviceIDHash,ExpId,datetime,date
4856,OffersScreenAppear,214966247576341063,248,2019-08-01 07:02:56,2019-08-01
4891,PaymentScreenSuccessful,214966247576341063,248,2019-08-01 07:04:00,2019-08-01
4893,CartScreenAppear,214966247576341063,248,2019-08-01 07:04:01,2019-08-01
4905,CartScreenAppear,214966247576341063,248,2019-08-01 07:04:18,2019-08-01
24257,PaymentScreenSuccessful,214966247576341063,248,2019-08-01 15:44:18,2019-08-01
24268,PaymentScreenSuccessful,214966247576341063,248,2019-08-01 15:44:28,2019-08-01
24269,CartScreenAppear,214966247576341063,248,2019-08-01 15:44:29,2019-08-01
53981,PaymentScreenSuccessful,214966247576341063,248,2019-08-02 12:46:40,2019-08-02
53984,CartScreenAppear,214966247576341063,248,2019-08-02 12:46:41,2019-08-02
53990,OffersScreenAppear,214966247576341063,248,2019-08-02 12:46:49,2019-08-02
